# Term Project Milestone 2 – Data Preparation (Revised)

**Student:** Anil Bammidi

**Revision note:** The first version kept `Height` and `Weight` as features and added BMI, which is computed from them. The target `NObeyesdad` is defined from BMI, so those inputs let the models recover the label directly. This revision removes `Height`, `Weight`, and the BMI feature from the model inputs and reports results on that basis.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import numpy as np
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Load data
DATA_PATH = "ObesityDataSet_raw_and_data_sinthetic.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nTarget classes:", sorted(df["NObeyesdad"].unique()))


## Milestone 1: Data Understanding and Initial Exploration
This section shows basic exploration of the obesity dataset: target distribution, summary statistics, and quick numeric relationships.


In [ ]:
# Basic info and missing values check
display(df.describe(include="all").T.head(20))
print("\nMissing values by column:")
display(df.isnull().sum().sort_values(ascending=False))


In [ ]:
# Target distribution
df.columns = df.columns.str.strip()
plt.figure(figsize=(10,4))
df["NObeyesdad"].value_counts().plot(kind="bar")
plt.title("Obesity Risk Class Distribution (Target)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Quick look at numeric distributions
num_cols = df.select_dtypes(include=["int64","float64"]).columns.tolist()
df[num_cols].hist(figsize=(12,8))
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix (numeric only)
num_cols = df.select_dtypes(include=["int64","float64"]).columns

corr = df[num_cols].corr()

plt.figure(figsize=(8,6))
plt.imshow(corr, aspect="auto")
plt.title("Correlation Matrix")
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar()
plt.tight_layout()
plt.show()


## Milestone 2: Data Preparation
In this milestone, the dataset is cleaned and prepared for modeling:
1. Drop duplicates
2. Validate values (Age > 0)
3. Remove leakage: drop `Height` and `Weight` (and do not derive BMI), because the target is defined from BMI
4. One-hot encode categorical variables and impute missing values inside the pipeline
5. Split into train/test before fitting any preprocessing (to avoid leakage)

In [ ]:
# Step 1: Remove duplicates
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Step 1 - Dropped duplicates: {before-after} removed, {after} rows remain.")


Leakage removal: the target `NObeyesdad` is assigned from BMI, which is computed from height and weight. Keeping `Height` or `Weight` (or a BMI feature built from them) would let the models reproduce the label rather than predict it from behavioral and demographic factors. These columns are excluded from the model inputs. The remaining features (age, gender, eating habits, activity, transportation, and so on) are the only inputs. Duplicate rows were removed to avoid bias in training.

In [ ]:
# Step 2: Validate values (keep only realistic positive ages)
before = df.shape[0]
df = df[df["Age"] > 0]
after = df.shape[0]
print(f"Step 2 - Filter invalid Age: {before-after} removed, {after} rows remain.")

In [ ]:
# Step 3: Leakage removal - drop height, weight, and any BMI feature from the inputs
LEAKAGE_COLS = ["Height", "Weight"]
print("Columns excluded from model inputs:", LEAKAGE_COLS)

The dataset contains minimal missing values. If missing values were present, numerical features would be filled using the median and categorical features using the mode. Dropping rows was avoided to preserve data and prevent bias.

### Train/Test Split (avoid data snooping)
We split the dataset **before** fitting any preprocessing steps. All imputers/encoders/scalers are trained only on the training set.


In [ ]:
# Split features/target (Height and Weight excluded to avoid leakage)
target = "NObeyesdad"
X = df.drop(columns=[target] + LEAKAGE_COLS)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

To avoid data leakage, preprocessing steps such as scaling and encoding were applied only to the training data using a pipeline. The test set was not used during training.

### Preprocessing Pipeline
- Numeric: median imputation + standard scaling
- Categorical: most frequent imputation + one-hot encoding


In [ ]:
# Identify numeric and categorical columns
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

# Build preprocessing
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


## Model Building and Evaluation
We train multiple supervised classification models and evaluate them on the held-out test set.
Metrics reported:
- Accuracy
- Macro F1 (handles class imbalance)
- Confusion Matrix
- Classification Report


In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    print(f"\n{name}")
    print("-"*len(name))
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(f1, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, preds))

    cm = confusion_matrix(y_test, preds, labels=sorted(y_test.unique()))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y_test.unique()))
    disp.plot(xticks_rotation=90)
    plt.title(f"Confusion Matrix: {name}")
    plt.tight_layout()
    plt.show()

    return acc, f1


In [ ]:
# Model 1: Multinomial Logistic Regression (baseline)
logreg = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

acc_lr, f1_lr = evaluate_model("Logistic Regression (Multinomial)", logreg, X_train, y_train, X_test, y_test)


In [ ]:
# Model 2: Random Forest
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])

acc_rf, f1_rf = evaluate_model("Random Forest", rf, X_train, y_train, X_test, y_test)


In [ ]:
# Model 3: Gradient Boosting (often strong on tabular data)
gb = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", GradientBoostingClassifier(random_state=42))
])

acc_gb, f1_gb = evaluate_model("Gradient Boosting", gb, X_train, y_train, X_test, y_test)


### Cross-Validation (Training Only)
We also run 5-fold cross-validation on the training set (no test leakage) to estimate stability.


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [
    ("LogReg", logreg),
    ("RandomForest", rf),
    ("GradBoost", gb)
]:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    print(f"{name} CV Accuracy: mean={scores.mean():.4f}, std={scores.std():.4f}")


### Feature Importance (Random Forest)
Random Forest provides feature importance values. After preprocessing (one-hot), we map importances back to feature names.


In [ ]:
# Fit RF pipeline to get feature importances
rf.fit(X_train, y_train)

# Get feature names after preprocessing
feature_names = rf.named_steps["prep"].get_feature_names_out()
importances = rf.named_steps["clf"].feature_importances_

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(15)

print("Top 15 Important Features (Random Forest):")
display(imp_df)

plt.figure(figsize=(10,4))
plt.bar(range(len(imp_df)), imp_df["importance"])
plt.xticks(range(len(imp_df)), imp_df["feature"], rotation=90)
plt.title("Top 15 Feature Importances (Random Forest)")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


## Final Summary (Revised Submission Write-Up)
- The dataset contains 2111 rows (2087 after removing 24 duplicates) and 16 predictors plus the multi-class target `NObeyesdad`.
- `Height` and `Weight` are excluded from the model inputs because the target is defined from BMI. The first version of this notebook included them and reported higher accuracy, which was driven by that leakage.
- Multiple models were trained and evaluated on the held-out test set: Multinomial Logistic Regression, Random Forest, and Gradient Boosting. Results are in the cells above.
- Accuracy and macro F1 are reported with confusion matrices. Five-fold cross-validation on the training set is included to check stability.
- Random Forest feature importances show which behavioral and demographic factors contribute most without the BMI inputs.
- The notebook was run on the revised feature set; see the project README for the reported results.